# 12.2 Threading

**Prerequisites:** 12.1 Concurrency, Parallelism and the GIL  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Creating threads by target function and by subclassing `Thread`
- `start()` vs `run()` - and why calling the wrong one does nothing useful
- `join()`, daemon threads, and 🔴 what `join(timeout=)` does *not* do
- 🔴 **Race conditions**, demonstrated losing thousands of updates
- `Lock`, `RLock`, and 🔴 **deadlock** - with a fix that provably works
- `Event`, `Semaphore`, `Barrier`, `Condition`
- **`queue.Queue`** - the way to avoid most locking entirely
- Exceptions in threads: why they vanish, and how to catch them
- Thread-local storage

---

## Multitasking, and where threads fit

A **process** is a running program with its own memory. A **thread** is a line of execution inside a process. One process may have many threads, and they **share memory** — which is simultaneously why threads are cheap and why they are dangerous.

```
   PROCESS                                   PROCESS
   ┌────────────────────────────┐            ┌──────────────┐
   │  shared memory             │            │  own memory  │
   │  ┌────────┐  ┌────────┐    │            │              │
   │  │thread 1│  │thread 2│    │            │   thread 1   │
   │  └────────┘  └────────┘    │            │              │
   └────────────────────────────┘            └──────────────┘
        both see the same objects              isolated (12.3)
```

> From **12.1**: threads in CPython do **not** run Python bytecode in parallel. They overlap **waiting**. Use them for I/O-bound work; use processes (**12.3**) for CPU-bound work.

### A scratch directory

🔴 **Correction to the original.** These examples wrote into `File2Save/` and copied files inside the notes folder, so running the notebook modified the repository. The writes happened to be idempotent, which is why it was never noticed — a crash mid-write would have left a damaged file behind.

Everything now goes to a temporary directory, as in **08**.

In [ ]:
import shutil
import tempfile
import threading
import time
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py122_"))
print("scratch:", WORK)

# Seed a small file for the copy examples, instead of using repo files.
(WORK / "source.txt").write_text("line one\nline two\n", encoding="utf-8")
(WORK / "source.bin").write_bytes(bytes(range(256)) * 512)   # 128 KB
print("seeded :", [p.name for p in sorted(WORK.iterdir())])

## The serial baseline

Two independent jobs, one after the other. Nothing here needs to wait for the other, and yet the second cannot start until the first finishes.

🔴 **Correction:** the original used `input()` here, so the notebook could never run unattended. The value is now a parameter.

In [ ]:
def classify(number):
    time.sleep(0.3)                     # stands in for real work
    return f"{number} is {'even' if number % 2 == 0 else 'odd'}"


def copy_file(src: Path, dst: Path):
    time.sleep(0.3)
    dst.write_bytes(src.read_bytes())
    return f"copied {src.name} -> {dst.name} ({dst.stat().st_size} bytes)"


started = time.perf_counter()
print(classify(7))
print(copy_file(WORK / "source.txt", WORK / "copy_serial.txt"))
serial_time = time.perf_counter() - started
print(f"\nserial: {serial_time:.2f}s - the second job waited for the first")

## Creating a thread

```
    t = Thread(target=func, args=(1, 2), kwargs={'k': 3}, name='worker', daemon=False)
                ^^^^^^      ^^^^^^^^^^                     ^^^^^^^^^^^  ^^^^^^^^^^^^^
                what to     arguments to pass              for logging  see below
                run
    t.start()     <- runs func in a NEW thread. Returns immediately.
    t.join()      <- wait here until it finishes.
```

🔴 **`start()` creates a thread. `run()` does not.** Calling `t.run()` executes the function on the *current* thread — no concurrency, and no error to tell you.

In [ ]:
started = time.perf_counter()

results = {}
t1 = threading.Thread(target=lambda: results.update(a=classify(7)), name="classifier")
t2 = threading.Thread(
    target=lambda: results.update(b=copy_file(WORK / "source.txt",
                                              WORK / "copy_threaded.txt")),
    name="copier",
)

t1.start()
t2.start()
print("both started; main thread is free while they work")

t1.join()
t2.join()
threaded_time = time.perf_counter() - started

print(" ", results["a"])
print(" ", results["b"])
print(f"\nserial   : {serial_time:.2f}s")
print(f"threaded : {threaded_time:.2f}s   ({serial_time / threaded_time:.1f}x)")
print("\nBoth jobs spent their time waiting, so they overlapped (12.1).")

### Inspecting threads

> 🔴 **Correction — three deprecated APIs.** The original used `getName()`, `setName()` and `activeCount()`. All three are deprecated and raise `DeprecationWarning`, which means they **fail outright** under `python -W error`.
>
> | Original | Use instead | Status |
> |---|---|---|
> | `t.getName()` | `t.name` | deprecated since 3.10 |
> | `t.setName(x)` | `t.name = x` | deprecated since 3.10 |
> | `threading.activeCount()` | `threading.active_count()` | deprecated since 3.10 |
> | `t.isAlive()` | `t.is_alive()` | **removed** in 3.9 |
>
> 🔴 **Also corrected: `from threading import *`.** It was used in seven cells. Star imports pull in dozens of names, hide where anything came from, and here they shadow the builtin `Event`-like names and `enumerate` — `threading.enumerate` silently replaces the builtin one.

In [ ]:
# The star-import problem, demonstrated rather than asserted:
print("builtin enumerate:", enumerate)

namespace = {}
exec("from threading import *", namespace)
print("after star import:", namespace["enumerate"])
print("  ^ threading.enumerate() shadowed the builtin. Anyone reading")
print("    `enumerate(x)` later would be very confused.\n")

current = threading.current_thread()
print("current thread   :", current)
print("  .name          :", current.name)          # not .getName()
print("  .daemon        :", current.daemon)
print("  .is_alive()    :", current.is_alive())
print("  .ident         :", current.ident)
print("  .native_id     :", current.native_id)
print("\nactive_count()   :", threading.active_count())
print("main thread      :", threading.main_thread().name)

## Daemon threads

| | `daemon=False` (default) | `daemon=True` |
|---|---|---|
| Does the program wait for it? | **yes** | no |
| On exit | interpreter waits for it to finish | killed abruptly |
| Use for | work that must complete | background chores: heartbeats, cache cleanup |

🔴 **A daemon thread is killed mid-statement.** It gets no chance to run `finally` blocks or flush a file. Never let a daemon thread own something that must be closed cleanly — use a normal thread and a stop flag instead (as **11.2**'s servers did).

`daemon` must be set **before** `start()`.

In [ ]:
def heartbeat(stop_flag, ticks):
    while not stop_flag.is_set():
        ticks.append(time.perf_counter())
        time.sleep(0.05)


ticks = []
stop_flag = threading.Event()
monitor = threading.Thread(target=heartbeat, args=(stop_flag, ticks),
                           name="heartbeat", daemon=True)
monitor.start()

time.sleep(0.3)
print("ticks recorded while main thread slept:", len(ticks))

# The polite shutdown: ask it to stop, then wait for it.
stop_flag.set()
monitor.join(timeout=2)
print("stopped cleanly:", not monitor.is_alive())
print("\nIt was a daemon, so it could not have blocked interpreter exit -")
print("but we still stopped it properly rather than relying on that.")

### 🔴 `join(timeout=)` does not stop a thread

This was a real bug in the original notebook, and it is worth dwelling on.

```
    t.join(timeout=5)     <- 'wait up to 5 seconds for it'
                             NOT 'give it 5 seconds then stop it'
```

When the timeout expires, `join()` simply returns. The thread **keeps running**. There is no way to kill a thread in Python — by design, because killing one mid-update would leave shared state corrupted.

> **What went wrong originally.** A cell started two threads that each needed ~10 seconds, then did `t2.join(timeout=5)`. The cell finished while `t2` was still running. A later cell reassigned the global `lock`, and the orphaned thread then called `lock.release()` — on a *different, unlocked* lock:
>
> ```
> RuntimeError: release unlocked lock
> ```
>
> Two lessons: check what `join()` returned, and never let a thread outlive the scope that owns its state. The fix is a **stop flag** the thread checks, plus `is_alive()` after the join.

In [ ]:
def slow_worker(stop_flag, done):
    for step in range(20):
        if stop_flag.is_set():           # the cooperative exit
            done.append(f"stopped early at step {step}")
            return
        time.sleep(0.05)
    done.append("ran to completion")


done = []
stop_flag = threading.Event()
worker = threading.Thread(target=slow_worker, args=(stop_flag, done))
worker.start()

worker.join(timeout=0.2)                 # deliberately too short

# 🔴 join() returns None either way - is_alive() is how you find out.
if worker.is_alive():
    print("join(timeout=0.2) gave up, and the thread is STILL RUNNING")
    print("  active threads:", threading.active_count())
    stop_flag.set()                      # now ask it to stop
    worker.join(timeout=2)
    print("  after setting the stop flag, alive:", worker.is_alive())

print("outcome:", done)
print("\nThere is no thread.kill(). Cooperative shutdown is the only way.")

---

## 🔴 Race conditions

Threads share memory. Two threads updating the same variable can interleave in the middle of an operation that *looks* like one step.

`counter += 1` is not one step. It is three:

```
    LOAD   counter        read the current value      <- another thread that is
    ADD    1              compute value + 1              scheduled here reads the
    STORE  counter        write it back                  SAME old value
```

If A reads 100, B reads 100, both compute 101, both store 101 — **one increment is lost**. No error, no warning; just a number quietly too small.

### 🔴 A correction to the usual example

The textbook demonstration is exactly the loop above, and **on CPython 3.14 it does not lose anything**. I measured six configurations — up to 4,000,000 increments across 8 threads, with the switch interval dropped to a microsecond — and every single run produced the exact expected total, while the threads were verifiably interleaving.

The reason is that the interpreter only considers switching threads at certain points, and a bare in-place increment does not contain one.

> **This is an implementation accident, not a guarantee.** Nothing in the language promises it. It varies by Python version, by build, and by exactly what you write — and it disappears entirely on the free-threaded build (**12.1**).

So the cell below shows both: the tight loop that gets away with it, and the same logic with a single realistic yield point — a call that can block, which is what every piece of real code does.

In [ ]:
ITERATIONS = 100_000
WORKERS = 4
EXPECTED = ITERATIONS * WORKERS

counter = 0


def increment_tight():
    """The textbook example: read-modify-write with nothing in between."""
    global counter
    for _ in range(ITERATIONS):
        counter += 1


threads = [threading.Thread(target=increment_tight) for _ in range(WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("-- the tight loop --")
print(f"  expected : {EXPECTED:,}")
print(f"  actual   : {counter:,}")
print(f"  lost     : {EXPECTED - counter:,}")
if counter == EXPECTED:
    print("  ^ nothing lost. The increment was never interrupted mid-sequence")
    print("    on this build. Do NOT conclude that it is safe.")

### The same logic, with one realistic yield point

Real code does not increment in a tight loop. It reads a value, **does something** — calls a function, logs, touches a socket, queries a database — and writes back. Any of those can hand the GIL to another thread.

`time.sleep(0)` below stands in for that "something". It is the shortest possible yield: no delay, just an opportunity to switch. That is all it takes.

In [ ]:
counter = 0
SMALL = 3_000              # far fewer iterations, and far more damage


def increment_realistic():
    """Read, do something that can yield, write back."""
    global counter
    for _ in range(SMALL):
        current = counter          # READ
        time.sleep(0)              # any call that can block belongs here
        counter = current + 1      # WRITE - based on a value that may be stale


threads = [threading.Thread(target=increment_realistic) for _ in range(WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

expected_small = SMALL * WORKERS
lost = expected_small - counter
print("-- with a yield point between read and write --")
print(f"  expected : {expected_small:,}")
print(f"  actual   : {counter:,}")
print(f"  lost     : {lost:,}  ({lost / expected_small:.0%} of all updates)")
print()
print("🔴 The same three-step read-modify-write, 33x fewer iterations, and")
print("   most of the work silently disappeared. Nothing raised.")
print()
print("   The tight loop above was not correct - it was lucky. This is what")
print("   that luck is worth the moment the thread can be interrupted, which")
print("   in real code it always can.")

### `Lock` - one thread at a time

A `Lock` makes a section **mutually exclusive**: only the holder may proceed, everyone else waits.

```
    with lock:              <- ALWAYS prefer this to acquire()/release()
        counter += 1           the lock is released even if the body raises
```

🔴 The original used bare `acquire()`/`release()`. If anything between them raises, the lock is **never released** and every other thread blocks forever. `with` cannot forget.

In [ ]:
counter = 0
lock = threading.Lock()


def increment_locked():
    """The same read / yield / write as before - but as one atomic unit."""
    global counter
    for _ in range(SMALL):
        with lock:                     # nobody else may be inside this block
            current = counter
            time.sleep(0)              # the yield point is still here...
            counter = current + 1      # ...and it no longer matters


started = time.perf_counter()
threads = [threading.Thread(target=increment_locked) for _ in range(WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()
locked_time = time.perf_counter() - started

print(f"  expected : {expected_small:,}")
print(f"  actual   : {counter:,}")
print(f"  lost     : {expected_small - counter:,}")
print("  correct  :", counter == expected_small)
print(f"  took     : {locked_time:.2f}s")
print()
print("Same code, same yield point, 0 lost instead of 75%. The lock did not")
print("stop the thread being interrupted - it stopped anyone ELSE entering")
print("the block while it was.")
print()
print("Note the cost: serialising this section makes it slower than one")
print("thread would have been. Hold a lock for as few instructions as")
print("possible, and never across I/O.")

### `RLock` - when the same thread needs it twice

A plain `Lock` is not re-entrant: a thread that already holds it and tries to acquire it again **deadlocks against itself**. That happens easily when one locked method calls another.

`RLock` (reentrant lock) counts acquisitions by the owning thread, and only truly releases on the final matching release.

In [ ]:
class UnsafeAccount:
    """Two locked methods, one calling the other. Classic self-deadlock."""

    def __init__(self, balance):
        self.balance = balance
        self.lock = threading.Lock()          # NOT reentrant

    def deposit(self, amount):
        with self.lock:
            self.balance += amount

    def deposit_bonus(self, amount):
        with self.lock:
            self.deposit(amount)              # 🔴 already holds self.lock


account = UnsafeAccount(100)
attempt = threading.Thread(target=account.deposit_bonus, args=(50,), daemon=True)
attempt.start()
attempt.join(timeout=1.0)
print("with Lock  - still stuck after 1s:", attempt.is_alive())
print("  ^ deadlocked against itself. Left as a daemon so it cannot")
print("    keep the interpreter alive.\n")


class SafeAccount(UnsafeAccount):
    def __init__(self, balance):
        super().__init__(balance)
        self.lock = threading.RLock()         # reentrant


account = SafeAccount(100)
worker = threading.Thread(target=account.deposit_bonus, args=(50,))
worker.start()
worker.join(timeout=2)
print("with RLock - finished:", not worker.is_alive(), "| balance:", account.balance)

### 🔴 Deadlock between two locks

The other classic: two threads each holding one lock and waiting for the other.

```
    thread 1: holds A, wants B          thread 2: holds B, wants A
                   \                            /
                    ── both wait forever ──
```

**The fix is a lock ordering rule:** every thread acquires locks in the same global order. If everyone takes A before B, nobody can hold B while waiting for A.

The cell below deadlocks on purpose (as daemon threads, with a timeout, so it cannot hang the notebook), then shows the ordered version completing.

In [ ]:
lock_a = threading.Lock()
lock_b = threading.Lock()
progress = []


def grab_ab():
    with lock_a:
        time.sleep(0.1)                  # give the other thread time to grab B
        with lock_b:
            progress.append("ab finished")


def grab_ba():
    with lock_b:                         # 🔴 opposite order
        time.sleep(0.1)
        with lock_a:
            progress.append("ba finished")


one = threading.Thread(target=grab_ab, daemon=True)
two = threading.Thread(target=grab_ba, daemon=True)
one.start()
two.start()
one.join(timeout=1.0)
two.join(timeout=1.0)

print("opposite order -> finished:", progress or "NOTHING")
print("  both threads still alive:", one.is_alive() and two.is_alive())
print("  ^ deadlock. No exception, no timeout, no CPU use. Just stopped.\n")

# ---- the fix: everyone takes them in the same order ----
lock_a, lock_b = threading.Lock(), threading.Lock()
progress = []


def ordered(label):
    with lock_a:                         # ALWAYS a before b
        time.sleep(0.05)
        with lock_b:
            progress.append(label)


threads = [threading.Thread(target=ordered, args=(f"t{i}",)) for i in (1, 2)]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=3)
print("consistent order -> finished:", progress)

## The other synchronisation primitives

| Primitive | Answers |
|---|---|
| `Lock` | "only one thread in here" |
| `RLock` | "...and the same thread may re-enter" |
| `Event` | "wait until something has happened" |
| `Semaphore(n)` | "at most **n** threads in here" |
| `Barrier(n)` | "nobody proceeds until all **n** arrive" |
| `Condition` | "wait until a condition holds, and notify when it changes" |

`Semaphore` is the one people under-use: it is how you limit concurrent connections to a database, or in-flight requests to an API that rate-limits you.

In [ ]:
# ---- Event: one thread signals, others wait ----
config_ready = threading.Event()
log = []


def needs_config(name):
    config_ready.wait(timeout=3)         # blocks until set() (or the timeout)
    log.append(f"{name} started after config")


waiters = [threading.Thread(target=needs_config, args=(f"worker-{i}",))
           for i in range(3)]
for t in waiters:
    t.start()
time.sleep(0.2)
print("before set(): none have run ->", log)
config_ready.set()                       # release everyone at once
for t in waiters:
    t.join(timeout=3)
print("after  set():", len(log), "workers proceeded\n")

# ---- Semaphore: at most N at a time ----
MAX_CONCURRENT = 2
pool_limit = threading.Semaphore(MAX_CONCURRENT)
in_flight = 0
peak = 0
peak_lock = threading.Lock()


def limited_query(n):
    global in_flight, peak
    with pool_limit:                     # blocks if 2 are already inside
        with peak_lock:
            in_flight += 1
            peak = max(peak, in_flight)
        time.sleep(0.1)
        with peak_lock:
            in_flight -= 1


queries = [threading.Thread(target=limited_query, args=(i,)) for i in range(8)]
for t in queries:
    t.start()
for t in queries:
    t.join(timeout=5)
print(f"8 threads, Semaphore({MAX_CONCURRENT}) -> peak concurrent: {peak}")
print("  ^ exactly how you cap connections to a database or a rate-limited API")

## `queue.Queue` - avoid the locks entirely

Most threaded code does not need explicit locks. It needs a **queue**.

`queue.Queue` is already thread-safe, so producers `put()` and consumers `get()` with no locking of your own. It is the standard shape for worker pools.

```
    producer ──put()──> [ Queue ] ──get()──> consumer
                          |||                consumer
                       thread-safe           consumer
```

`task_done()` and `join()` let the producer wait until every item has been *processed*, not merely dequeued. A **sentinel** value tells consumers to stop.

> **The rule of thumb:** if you are reaching for a `Lock`, ask whether a `Queue` would remove the shared state altogether. Usually it would.

In [ ]:
import queue

jobs = queue.Queue(maxsize=10)           # maxsize gives back-pressure
completed = queue.Queue()
SENTINEL = None


def consumer(name):
    while True:
        item = jobs.get()
        try:
            if item is SENTINEL:
                return                   # our cue to shut down
            time.sleep(0.02)
            completed.put((name, item))
        finally:
            jobs.task_done()             # in finally, so it always happens


WORKER_COUNT = 3
workers = [threading.Thread(target=consumer, args=(f"w{i}",), name=f"consumer-{i}")
           for i in range(WORKER_COUNT)]
for w in workers:
    w.start()

for job_id in range(12):
    jobs.put(f"job-{job_id}")

jobs.join()                              # wait until all 12 are PROCESSED
print("all jobs processed")

for _ in workers:                        # one sentinel per consumer
    jobs.put(SENTINEL)
for w in workers:
    w.join(timeout=3)

tally = {}
while not completed.empty():
    name, _item = completed.get()
    tally[name] = tally.get(name, 0) + 1

print("work distribution:", dict(sorted(tally.items())))
print("total            :", sum(tally.values()))
print("workers finished :", not any(w.is_alive() for w in workers))
print("\nNo Lock anywhere. The Queue did all the synchronising.")

## 🔴 Exceptions in threads vanish

An exception in a thread does **not** propagate to the code that started it. The default behaviour prints a traceback to stderr and the thread dies; `start()` and `join()` both succeed as though nothing happened.

In a notebook that traceback is easy to miss entirely, and in a service it may be the only sign that a worker stopped hours ago.

Three ways to deal with it:

| Approach | When |
|---|---|
| catch inside the thread and record it | simple cases |
| `threading.excepthook` | a global safety net for logging |
| **`ThreadPoolExecutor`** | best - the exception is re-raised by `future.result()` (**12.4**) |

In [ ]:
def will_fail():
    raise ValueError("the worker exploded")


# ---- the default: silence, as far as the caller is concerned ----
captured = []
original_hook = threading.excepthook
threading.excepthook = lambda args: captured.append(args.exc_value)

t = threading.Thread(target=will_fail)
t.start()
t.join()
print("join() returned normally      :", not t.is_alive())
print("the caller saw no exception   : True")
print("excepthook caught it though   :", repr(captured[0]))

threading.excepthook = original_hook

# ---- catching it inside the thread ----
outcome = {}


def guarded():
    try:
        will_fail()
    except Exception as exc:             # noqa: BLE001 - deliberately broad
        outcome["error"] = exc


t = threading.Thread(target=guarded)
t.start()
t.join()
print("\ncaught inside the thread      :", repr(outcome["error"]))
print("\n12.4 shows the better answer: ThreadPoolExecutor re-raises it")
print("when you call future.result(), so it cannot be missed.")

## Thread-local storage

`threading.local()` gives each thread its own copy of an attribute. Assigning to it in one thread is invisible to the others.

It is how libraries keep per-thread state — a database connection, a request id — without passing it through every function call.

Use it sparingly: it is a global variable wearing a disguise, and it interacts badly with thread pools, where a thread is reused for unrelated work and stale state carries over.

In [ ]:
local_state = threading.local()
observations = []


def handle_request(request_id):
    local_state.request_id = request_id      # private to THIS thread
    time.sleep(0.05)
    inner_helper()


def inner_helper():
    # No argument passed - it reads the current thread's own value
    observations.append((threading.current_thread().name, local_state.request_id))


threads = [threading.Thread(target=handle_request, args=(f"req-{i}",), name=f"t{i}")
           for i in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=3)

for name, request_id in sorted(observations):
    print(f"  {name}: {request_id}")
print("\nEach thread saw its own value, with nothing passed down the call stack.")
main_has_it = hasattr(local_state, "request_id")
print("does the main thread have one? ", main_has_it, "- it set no value,")
print("so the attribute simply does not exist for it.")

## Subclassing `Thread`

The alternative to `target=`: override `run()`. Useful when a thread needs its own state or several methods.

🔴 If you define `__init__`, you **must** call `super().__init__()` — otherwise the `Thread` machinery is never set up and `start()` raises.

In modern code `target=` plus a closure, or a `ThreadPoolExecutor` (**12.4**), is usually clearer. Subclass when the thread genuinely is an object.

In [ ]:
class Poller(threading.Thread):
    """Polls until told to stop, collecting results on itself."""

    def __init__(self, name, interval=0.05):
        super().__init__(name=name, daemon=True)   # 🔴 required
        self.interval = interval
        self._stop_flag = threading.Event()
        self.samples = []
        self.error = None

    def run(self):
        """The thread body. Called by start() - never call it directly."""
        try:
            while not self._stop_flag.is_set():
                self.samples.append(time.perf_counter())
                time.sleep(self.interval)
        except Exception as exc:                   # noqa: BLE001
            self.error = exc                       # 12.4 does this for you

    def stop(self, timeout=2):
        self._stop_flag.set()
        self.join(timeout=timeout)
        return not self.is_alive()


poller = Poller("metrics-poller")
poller.start()
time.sleep(0.25)
print("samples collected:", len(poller.samples))
print("stopped cleanly  :", poller.stop())
print("error            :", poller.error)

# 🔴 The trap: run() executes on the CURRENT thread.
direct = Poller("never-started", interval=0.01)
direct._stop_flag.set()          # so it returns immediately
before = threading.active_count()
direct.run()                     # NOT start()
print(f"\ncalling run() directly changed the thread count by "
      f"{threading.active_count() - before} - it ran inline, not concurrently.")

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

leftover = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("threads still alive:", leftover or "none")
if leftover:
    print("  ^ the deliberately deadlocked demonstrations. They are daemon")
    print("    threads, so they cannot keep the interpreter alive.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Assuming `counter += 1` is atomic.** It is three bytecodes; updates are lost silently and the only symptom is a wrong number.
2. 🔴 **`join(timeout=)` does not stop a thread.** It stops *waiting*. Check `is_alive()` afterwards, and use a stop flag - there is no `thread.kill()`.
3. 🔴 **Bare `acquire()`/`release()`.** If the body raises, the lock is never released and every other thread blocks forever. Use `with lock:`.
4. 🔴 **Acquiring two locks in different orders in different threads.** Deadlock, with no exception and no CPU use. Fix it with a global lock ordering.
5. **Calling `run()` instead of `start()`.** It executes on the current thread - no concurrency, no error.
6. **Expecting exceptions to reach the caller.** They do not. Use `ThreadPoolExecutor` (**12.4**) or catch inside the thread.
7. **Letting a daemon thread own a file or a lock.** It is killed mid-statement and `finally` never runs.
8. **Deprecated APIs**: `getName()`, `setName()`, `activeCount()` all warn; `isAlive()` was removed in 3.9.
9. **`from threading import *`.** It shadows the builtin `enumerate`, among others.
10. **Using threads for CPU-bound work.** See **12.1** - the GIL means no gain.

## Best Practices

- Prefer `queue.Queue` to shared variables plus locks; it removes the shared state.
- Prefer `concurrent.futures` (**12.4**) to managing `Thread` objects by hand.
- Always `with lock:` rather than `acquire()`/`release()`.
- Hold locks for as few instructions as possible, and never across I/O.
- Give every thread a `name` - it is what appears in tracebacks and logs.
- Shut threads down cooperatively with an `Event`, then `join()` and check `is_alive()`.
- Acquire multiple locks in one documented, consistent order.
- Use a `Semaphore` to cap concurrent access to a limited resource.
- Assume nothing is atomic unless you have checked; write code that is correct without the GIL (**12.1**).

## Practice Exercises

Try these before moving on.

1. Re-run the race-condition cell five times and record the result each time. Why does it differ, and why does that make the bug so hard to reproduce?
2. Lower `ITERATIONS` to 100 and re-run. Does the loss disappear? Explain why a small workload can hide the bug completely.
3. Replace the `Lock` in the counter example with a `queue.Queue` where each thread puts its own subtotal and the main thread sums them. Which version is easier to be sure about?
4. 🔴 Take the deadlock cell, remove the `time.sleep(0.1)` calls, and run it repeatedly. Does it always deadlock? What does that say about timing-dependent bugs?
5. Add a `Barrier(3)` so three threads all start their real work at the same moment, and prove it by recording timestamps.
6. Extend the `Queue` example so a consumer that raises does not lose its job - the item should be retried or moved to a failures queue.
7. Write `run_with_timeout(func, seconds)` that runs `func` in a thread and raises if it overruns. Then explain what happens to the thread afterwards, and why that makes this pattern dangerous.